In [5]:
%pip install pandas
import pandas as pd

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 4.9 MB/s eta 0:00:01
     |████████████████████████████████| 348 kB 79.6 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 92.3 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 77.6 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [6]:
stats = pd.read_csv('data/advanced_stats.csv')
salary = pd.read_csv('data/player_salary.csv')

In [10]:
stats_clean = stats.copy()

# Drop columns we do not need for modeling
stats_clean = stats_clean.drop(columns=["Rk", "Player-additional"], errors="ignore")

# dropping duplicate player rows and empty columns
stats_clean = stats_clean.drop_duplicates(subset=["Player", "Team"], keep="first")
stats_clean = stats_clean.dropna(axis=1, how="all")

# Fill awards with empty string so it is consistent text data
if "Awards" in stats_clean.columns:
    stats_clean["Awards"] = stats_clean["Awards"].fillna("")

for col in ["Player", "Team", "Pos"]:
    if col in stats_clean.columns:
        stats_clean[col] = stats_clean[col].astype(str).str.strip()

salary_clean = salary.copy()

# Rename known placeholder columns first
salary_clean = salary_clean.rename(
    columns={
        "Unnamed: 0": "Rk",
        "Unnamed: 1": "Player",
        "Unnamed: 2": "Team",
    }
)

salary_clean = salary_clean[salary_clean["Rk"].astype(str) != "Rk"].copy()

# Rename salary columns to readable season names if present
season_map = {
    "Salary": "2025-26",
    "Salary.1": "2026-27",
    "Salary.2": "2027-28",
    "Salary.3": "2028-29",
    "Salary.4": "2029-30",
    "Salary.5": "2030-31",
    "Unnamed: 9": "Guaranteed",
    "-additional": "player_id",
}
salary_clean = salary_clean.rename(columns={k: v for k, v in season_map.items() if k in salary_clean.columns})

# keep only useful columns if they exist
keep_cols = [
    "Rk", "Player", "Team",
    "2025-26", "2026-27", "2027-28", "2028-29", "2029-30", "2030-31",
    "Guaranteed", "player_id",
]
salary_clean = salary_clean[[c for c in keep_cols if c in salary_clean.columns]]

# Parse currency strings to numeric
money_cols = [c for c in salary_clean.columns if c.startswith("20") or c == "Guaranteed"]
for col in money_cols:
    salary_clean[col] = (
        salary_clean[col]
        .astype("string")
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace({"": pd.NA})
    )
    salary_clean[col] = pd.to_numeric(salary_clean[col], errors="coerce")

# making the rank a number
if "Rk" in salary_clean.columns:
    salary_clean["Rk"] = pd.to_numeric(salary_clean["Rk"], errors="coerce")

# make the names text and remove empty names
if "Player" in salary_clean.columns:
    salary_clean["Player"] = salary_clean["Player"].astype("string").str.strip()
    salary_clean = salary_clean[salary_clean["Player"].notna() & (salary_clean["Player"] != "")]

# Basic text cleanup without converting nulls to literal "nan"
for col in ["Team", "player_id"]:
    if col in salary_clean.columns:
        salary_clean[col] = salary_clean[col].astype("string").str.strip()


stats_clean.to_csv("data/advanced_stats_clean.csv", index=False)
salary_clean.to_csv("data/player_salary_clean.csv", index=False)

print("Advanced stats cleaned shape:", stats_clean.shape)
print("Salary cleaned shape:", salary_clean.shape)
print("\nAdvanced stats preview:")
print(stats_clean.head())
print("\nSalary preview:")
print(salary_clean.head())

Advanced stats cleaned shape: (736, 28)
Salary cleaned shape: (530, 11)

Advanced stats preview:
            Player   Age Team Pos     G    GS      MP   PER    TS%   3PAr  \
0    Mikal Bridges  28.0  NYK  SF  82.0  82.0  3036.0  14.0  0.585  0.391   
1        Josh Hart  29.0  NYK  SG  77.0  77.0  2897.0  16.5  0.611  0.327   
2  Anthony Edwards  23.0  MIN  SG  79.0  79.0  2871.0  20.1  0.595  0.503   
3     Devin Booker  28.0  PHO  SG  75.0  75.0  2795.0  19.3  0.589  0.388   
4     James Harden  35.0  LAC  PG  79.0  79.0  2789.0  20.0  0.582  0.516   

   ...  USG%  OWS  DWS   WS  WS/48  OBPM  DBPM  BPM  VORP             Awards  
0  ...  19.6  3.7  2.0  5.7  0.090   0.4  -0.9 -0.5   1.2                     
1  ...  15.3  5.4  3.8  9.2  0.153   1.1   1.8  2.8   3.6                     
2  ...  31.4  4.6  3.8  8.4  0.140   4.4   0.0  4.3   4.6  MVP-7CPOY-3ASNBA2  
3  ...  29.3  6.1  0.3  6.4  0.111   2.8  -2.4  0.4   1.7                     
4  ...  29.6  4.0  4.3  8.3  0.143   3.5   0.